# Solutions — Capstone: Wanderbricks Platform (Hard 18)

In [ ]:
from pyspark.sql import functions as F, types as T
from pyspark.sql import Window

bookings   = spark.table("samples.wanderbricks.bookings")
users      = spark.table("samples.wanderbricks.users")
properties = spark.table("samples.wanderbricks.properties")
reviews    = spark.table("samples.wanderbricks.reviews")
payments   = spark.table("samples.wanderbricks.payments")
hosts      = spark.table("samples.wanderbricks.hosts")


## Solution 1 — Active Host Portfolio

In [ ]:
# Rank hosts by portfolio size
# hosts table has NO is_active or country columns
# country comes from properties; is_verified from hosts
result_1 = (
    properties
    .join(hosts.select("host_id", "name", "is_verified"), on="host_id")
    .groupBy("host_id", "name", "country", "is_verified")
    .agg(
        F.count("*").alias("property_count"),
        F.round(F.avg("base_price"), 2).alias("avg_base_price"),
        F.collect_set("property_type").alias("property_types"),
    )
    .orderBy(F.col("property_count").desc())
    .limit(20)
    .select("host_id", "name", "country", "is_verified", "property_count", "avg_base_price", "property_types")
)

## Solution 2 — Booking Revenue by Property Type and Country

In [ ]:
# Confirmed bookings; revenue per (property_type, country)
# hosts has no country column — country comes from properties
result_2 = (
    bookings
    .filter(F.col("status") == "confirmed")
    .join(properties.select("property_id", "property_type", "country"), on="property_id")
    .groupBy("property_type", "country")
    .agg(
        F.count("*").alias("total_bookings"),
        F.round(F.sum("total_amount"), 2).alias("total_revenue"),
        F.round(F.avg("total_amount"), 2).alias("avg_booking_value"),
        F.round(F.avg(F.datediff("check_out", "check_in")), 1).alias("avg_stay_nights"),
    )
    .orderBy(F.col("total_revenue").desc())
)

## Solution 3 — Guest Spending Behaviour

In [ ]:
# Top 20 guests by total spend; countries visited from properties.country
# hosts has no country column — use properties.country instead
result_3 = (
    bookings
    .filter(F.col("status") == "confirmed")
    .join(properties.select("property_id", "country"), on="property_id")
    .groupBy("user_id")
    .agg(
        F.count("*").alias("trips_taken"),
        F.round(F.sum("total_amount"), 2).alias("total_spend"),
        F.round(F.avg(F.datediff("check_out", "check_in")), 1).alias("avg_stay_nights"),
        F.collect_set("country").alias("countries_visited"),
    )
    .orderBy(F.col("total_spend").desc())
    .limit(20)
)

## Solution 4 — Review Quality by Property Type

In [ ]:
# Review quality per property type
# Expected columns: property_type, avg_rating, review_count, rated_property_count, quality_flag
result_4 = (
    reviews
    .filter(F.col("is_deleted") == False)
    .join(properties.select("property_id", "property_type"), on="property_id")
    .groupBy("property_type")
    .agg(
        F.round(F.avg("rating"), 2).alias("avg_rating"),
        F.count("*").alias("review_count"),
        F.countDistinct("property_id").alias("rated_property_count"),
    )
    .withColumn("quality_flag",
        F.when(F.col("avg_rating") < 3.5, "Needs Attention")
         .otherwise("Good")
    )
    .orderBy(F.col("avg_rating").desc())
)

## Solution 5 — Cancellation Intelligence

In [ ]:
# Per (property_type, country): cancellation rate; country from users
# Expected columns: property_type, country, total_bookings, cancelled_bookings, cancellation_rate_pct
result_5 = (
    bookings
    .join(users.select("user_id", F.col("country").alias("country")), on="user_id")
    .join(properties.select("property_id", "property_type"), on="property_id")
    .groupBy("property_type", "country")
    .agg(
        F.count("*").alias("total_bookings"),
        F.sum(F.when(F.col("status") == "cancelled", 1).otherwise(0)).alias("cancelled_bookings"),
    )
    .filter(F.col("total_bookings") >= 5)
    .withColumn("cancellation_rate_pct",
        F.round(F.col("cancelled_bookings") / F.col("total_bookings") * 100, 2)
    )
    .orderBy(F.col("cancellation_rate_pct").desc())
)

## Solution 6 — Seasonal Booking Patterns

In [ ]:
# Per (property_type, month): booking count; collect_set peak months (top 3)
w_rank = Window.partitionBy("property_type").orderBy(F.col("booking_count").desc())

monthly = (
    bookings
    .join(properties.select("property_id", "property_type"), on="property_id")
    .withColumn("month", F.month("check_in"))
    .groupBy("property_type", "month")
    .agg(F.count("*").alias("booking_count"))
    .withColumn("month_rank", F.rank().over(w_rank))
)

result_6 = (
    monthly
    .filter(F.col("month_rank") <= 3)
    .groupBy("property_type")
    .agg(F.collect_set("month").alias("peak_months"))
    .orderBy("property_type")
)


## Solution 7 — Host Performance Dashboard

In [ ]:
# Full dashboard: host → properties → bookings → payments → reviews
host_props = (
    properties
    .groupBy("host_id")
    .agg(F.count("*").alias("property_count"))
)

host_booking_rev = (
    bookings
    .filter(F.col("status") == "confirmed")
    .join(properties.select("property_id", "host_id"), on="property_id")
    .groupBy("host_id")
    .agg(
        F.round(F.sum("total_amount"), 2).alias("total_revenue"),
        F.sum(F.when(F.col("status") == "cancelled", 1).otherwise(0)).alias("cancelled_count"),
        F.count("*").alias("confirmed_bookings"),
    )
    .withColumn("cancellation_rate_pct",
        F.round(F.col("cancelled_count") / (F.col("confirmed_bookings") + F.col("cancelled_count")) * 100, 2)
    )
)

avg_guest_rating = (
    reviews.filter(F.col("is_deleted") == False)
    .join(properties.select("property_id", "host_id"), on="property_id")
    .groupBy("host_id")
    .agg(F.round(F.avg("rating"), 2).alias("avg_guest_rating"))
)

result_7 = (
    hosts
    .join(host_props,        on="host_id", how="left")
    .join(host_booking_rev,  on="host_id", how="left")
    .join(avg_guest_rating,  on="host_id", how="left")
    .select(
        "host_id", "name", "total_revenue",
        "avg_guest_rating", "cancellation_rate_pct"
    )
    .orderBy(F.col("total_revenue").desc())
)
